# UKB colocalization code

In [ ]:
cd /sc/arion/projects/ad-omics/haochen

zip -r UKB.zip rss_qc/genotypes/gtex_trimed_random/ukbb_unrelated_trimed

In [1]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/code/UKB_fine.R
# ============================================================
# UKB blood-trait GWAS fine-mapping with KG LD reference
#   - SuSiE-RSS with UKB-provided LD ("in-sample" panel)
#   - SuSiE-RSS with KG LD (external panel) + kriging QC
#   - base / EB (haochen build) / EB-mix (xuewei build)
#
# Usage: Rscript UKB_fine.R <IDX>
# ============================================================

# ---- batch index from command line ----
args <- commandArgs(trailingOnly = TRUE)
IDX  <- as.integer(args[1])
stopifnot(!is.na(IDX), IDX >= 1)

library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

# ============================================================
# Two susieR builds, each ISOLATED in its own environment so their
# identically-named functions (susie_rss + helpers) never clobber.
# Extracting $susie_rss keeps the function bound to its build's env,
# so its internal helper calls still resolve within that build.
# ============================================================
source_build <- function(dir) {
  rdir <- if (dir.exists(file.path(dir, "R"))) file.path(dir, "R") else dir
  env  <- new.env(parent = globalenv())
  for (f in list.files(rdir, full.names = TRUE, pattern = "\\.R$"))
    sys.source(f, envir = env)
  env
}

susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
susie_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/susieR_July17_2026_singleL/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

sketch_harmonize <- function(target_data, ref_variants, sketch) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(sketch))
  if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
  sketch2 <- sketch[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
  colnames(sketch2) <- qced$variants_id_original
  list(sumstat = qced, sketch = sketch2,
       flipped_prop = length(flip_index) / ncol(sketch2))
}

# max inter-CS correlation (NA when < 2 CS), against the LD the fit used
cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- config ----
sumstat_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/UKB_blood_real_data/"
snp_info_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/snp_info/"
out_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/result"
ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv"

maf.cut     <- 0.05
missing.cut <- 0.1
B_insample  <- 1e6      # in-sample LD is exact -> huge R_finite (EB captures only residual)
B_sketch    <- 10000    # finite-reference size for the KG / sketch panel

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

files <- list.files(sumstat_dir, pattern = ".summary_stats.rds", full.names = TRUE)

idx_start <- 15 * (IDX - 1) + 1
idx_end   <- min(15 * IDX, length(files))

# ============================================================
# Outer loop: regions
# ============================================================
for (i in idx_start:idx_end) {
  tryCatch({
    file      <- files[i]
    base_file <- basename(file)

    # ---- 1. parse region from filename ----
    parts     <- str_split(base_file, "\\.")[[1]]
    blood_chr <- parts[1]
    start     <- parts[2]
    end       <- parts[3]
    chrom     <- str_remove(str_split(blood_chr, "_")[[1]][2], "chr")
    region    <- paste0("chr", chrom, ":", start, "-", end)

    # skip-existing
    res_file <- file.path(out_dir, paste0("res_", region, ".rds"))
    if (file.exists(res_file)) next

    # ---- 2. load sumstats + sufficient stats ----
    suff_stat_file <- str_replace(base_file, "summary", "sufficient")
    sumstat_list   <- readRDS(file)
    suff_stat_list <- readRDS(file.path(sumstat_dir, suff_stat_file))

    if (ncol(sumstat_list$Z) == 0) next

    # dbsnp hg38 lookup (CHR/BP already lifted)
    dpsnp_match <- read_tsv(file.path(snp_info_dir, paste0("chrom_", chrom, "_snp.tsv")),
                            show_col_types = FALSE)

    snp_meta     <- sumstat_list$meta
    target_input <- snp_meta %>%
      left_join(dpsnp_match %>% select(SNP, BP), by = c("ID" = "SNP")) %>%
      select(CHR, BP, ID, REF, ALT, maf) %>%
      mutate(original_ID = paste0(ID, "_", REF)) %>%
      arrange(BP) %>%
      na.omit()

    start  <- min(target_input$BP)
    end    <- max(target_input$BP)
    region <- sprintf("chr%s:%d-%d", chrom, start, end)

    res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
    res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
    sketch_ref           <- res_sketch$ref_panel

    # ---- 3. trait loop ----
    res_susie <- list()

    for (tissue_idx in seq_len(ncol(sumstat_list$Z))) {

      tissue_name <- colnames(sumstat_list$Z)[tissue_idx]
      n           <- suff_stat_list$N
      z           <- sumstat_list$Z[, tissue_idx]

      message(sprintf("[%s]   trait: %s",
                      format(Sys.time(), "%Y-%m-%d %H:%M:%S"), tissue_name))

      # build pos-sorted sumstat with z
      snp_meta <- target_input
      z_df     <- tibble(z = z, original_ID = rownames(sumstat_list$Z))
      snp_meta <- left_join(snp_meta, z_df, by = "original_ID")
      colnames(snp_meta) <- c("chrom", "pos", "snp", "A2", "A1", "maf", "original_ID", "z")

      # ---- allele QC against KG reference ----
      harm <- sketch_harmonize(target_data  = snp_meta,
                               ref_variants = sketch_ref %>% select(-variant_id),
                               sketch       = res_sketch$LD_matrix)


      sketch_LD          <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)
      krig           <- kriging_fix(z = harm$sumstat$z, LD = sketch_LD , n = n)
      harm$sumstat$z <- krig$z

      if (ncol(harm$sketch) == 0) {
        message(sprintf("  No variants survive harmonization for trait %s - skipping", tissue_name))
        next
      }

      # ---- in-sample z / LD aligned to the harmonized variant set ----
      insample_z  <- sumstat_list$Z[harm$sumstat$original_ID, tissue_idx]
      insample_LD <- sumstat_list$LD[harm$sumstat$original_ID, harm$sumstat$original_ID]

      # sketch-panel LD (from the factor matrix) for CS-correlation diagnostics
      sketch_LD <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)

      # ==========================================================
      # Fits: base / EB (susie_hc)  +  EB-mix (susie_hc)
      # ==========================================================

      # ---- in-sample panel ----
      insample_res         <- safe_susie(susie_hc, "insample_res",
                                         z = insample_z, R = insample_LD, n = n)


      insample_eb_mix_res  <- safe_susie(susie_hc, "insample_eb_mix_res",
                                         z = insample_z, R = insample_LD, n = n,
                                         R_finite = B_insample, R_mismatch = "eb_mix")

      # ---- sketch / KG panel ----
      susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                         z = harm$sumstat$z, X = harm$sketch, n = n)

      susie_eb_mix_res         <- safe_susie(susie_hc, "susie_eb_mix_res",
                                         z = harm$sumstat$z, X = harm$sketch, n = n,
                                         R_finite = B_sketch, R_mismatch = "eb_mix")

      susie_eb_mix_res_new <- safe_susie(
          susie_xw, "susie_eb_mix_res_new",
          z = harm$sumstat$z, X = harm$sketch, n = n,
          R_finite = B_sketch, R_mismatch = "eb_mix", check_prior = FALSE
        )
        

      susie_ser_res        <- safe_susie(susie_hc, "susie_ser_res",
                                         z = harm$sumstat$z, X = harm$sketch, n = n,
                                         L = 1, max_iter = 1)

      # ---- max inter-CS correlation, one per fit ----
      max_cs_corr <- c(
        insample_res         = cs_maxcor(insample_res,         insample_LD),
        insample_eb_mix_res  = cs_maxcor(insample_eb_mix_res,  insample_LD),
        susie_rss_res        = cs_maxcor(susie_rss_res,        sketch_LD),
        susie_eb_mix_res         = cs_maxcor(susie_eb_mix_res,         sketch_LD),
        susie_eb_mix_res_new   = cs_maxcor(susie_eb_mix_res_new,   sketch_LD),
        susie_ser_res        = cs_maxcor(susie_ser_res,        sketch_LD)
      )

      res_susie[[tissue_name]] <- list(
        variant_ids          = colnames(harm$sketch),
        # --- fits ---
        insample_res         = insample_res,
        insample_eb_mix_res  = insample_eb_mix_res,
        susie_rss_res        = susie_rss_res,
        susie_eb_mix_res         = susie_eb_mix_res,
        susie_eb_mix_res_new     = susie_eb_mix_res_new,
        susie_ser_res        = susie_ser_res,
        # --- diagnostics / bookkeeping ---
        max_cs_corr          = max_cs_corr,
        sample_size          = n,
        flipped_prop         = harm$flipped_prop,
        original_file_path = file,
        kriging_outlier_number = krig$outlier_number,  
        region = region
      )
    }

    # ---- save per-region ----
    if (length(res_susie) == 0) {
      saveRDS("no traits processed",
              file.path(out_dir, paste0("nores_", region, ".rds")))
    } else {
      saveRDS(res_susie, res_file)
    }
  }, error = function(e) {
    cat(sprintf("[FAIL] region i=%d (%s): %s\n",
                i, basename(files[i]), conditionMessage(e)))
  })
}

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/code/UKB_fine.R


In [2]:
%%writefile  /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/code/submit_UKB_fine.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J "UKB_ske[1-65]"
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/log/UKB_ske.%J.%I.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/log/UKB_ske.%J.%I.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/
mkdir -p ${workdir}/log
code_dir="$workdir/code"
tmp_root="$workdir/tmp"

tmp_dir="$tmp_root/${LSB_JOBID}_${LSB_JOBINDEX}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

Rscript "$code_dir/UKB_fine.R" "$LSB_JOBINDEX"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/code/submit_UKB_fine.sh


In [ ]:
sb /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/code/submit_UKB_fine.sh

In [ ]:
[coloc]
parameter: in_dir = path
# directory containing res_*.rds files (per-gene SuSiE results)
parameter: out_dir = path("output")
# directory to write per-gene coloc_*.rds files
parameter: job_size = 50
parameter: walltime = "24h"
parameter: mem = "20G"
parameter: numThreads = 1
parameter: pp_thres = 0.8
# PP.H4 threshold for declaring colocalization / replication
parameter: container = ""

# discover one input file per gene and produce one output file per gene
input: [str(f) for f in path(in_dir).glob("res_*.rds")], group_by = 1
output: f'{out_dir:a}/coloc_{_input:bn}.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    suppressPackageStartupMessages({
      library(tidyverse)
      library(pecotmr)
      library(coloc)
    })

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    in_file  <- ${_input:r}
    out_file <- ${_output:r}
    gene     <- str_remove(basename(in_file), "^res_") %>% str_remove("\\.rds$")
    pp_thres <- ${pp_thres}
    cat(sprintf("[%s] gene: %s\n", format(Sys.time(), "%H:%M:%S"), gene))
    flush.console()



    sumstat_methods <- c(
      "insample_res",
      "insample_eb_mix_res",
      "susie_rss_res",
      "susie_eb_mix_res",
      "susie_eb_mix_res_new",
      "susie_ser_res"
    )

    # ---- diagnostic extraction --------------------------------------------

    flag_or_na <- function(x) if (is.null(x) || length(x) == 0) NA else as.logical(x)[1]

    get_diag <- function(fit) {
      diag <- fit$R_finite_diagnostics       # NULL for non-EB fits -> all NA
      list(
        B_corrected        = as.numeric(fit$B_corrected %||% NA_real_),
        lambda_bias        = as.numeric(fit$lambda_bias %||% NA_real_),
        Q_art              = as.numeric(diag$Q_art %||% NA_real_),
        D_CS_max           = if (!is.null(diag$bf_attenuation$cs_max))
                               suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE))
                             else NA_real_,
        max_abs_z          = if (!is.null(diag$ser_model$z))
                               suppressWarnings(max(abs(diag$ser_model$z), na.rm = TRUE))
                             else NA_real_,
        artifact_evaluable = flag_or_na(diag$artifact_evaluable),
        reliability_flag   = flag_or_na(diag$R_reliability_flag),
        sensitivity_flag   = flag_or_na(diag$R_sensitivity_flag),
        artifact_flag      = flag_or_na(fit$artifact_flag),
        prior_flag         = flag_or_na(fit$prior_variance_flag)
      )
    }

    coloc_pair <- function(insample, outsample) {
      colnames(insample$alpha)        <- colnames(outsample$alpha)
      colnames(insample$mu)           <- colnames(outsample$mu)
      colnames(insample$mu2)          <- colnames(outsample$mu2)
      colnames(insample$lbf_variable) <- colnames(outsample$lbf_variable)
      names(insample$X_column_scale_factors) <- names(outsample$X_column_scale_factors)
      names(insample$pip)             <- names(outsample$pip)
      coloc.susie(insample, outsample)
    }

    mcc_get <- function(mcc, method) {
          if (is.null(mcc) || is.null(names(mcc)) || !method %in% names(mcc))
            return(NA_real_)
          as.numeric(mcc[[method]])
    }

    one_record <- function(insample, outsample, gene, tissue, method, max_corr = NA_real_) {
      n_in  <- length(insample$sets$cs)
      n_out <- length(outsample$sets$cs)
      if (n_in == 0 && n_out == 0) return(NULL)

      dg <- get_diag(outsample)             # metrics describe the tested (outsample) fit

      rec <- tibble(
        gene = gene, tissue = tissue, method = method,
        n_insample_cs = n_in, n_outsample_cs = n_out,
        replicated_insample = NA_integer_, replicated_outsample = NA_integer_,
        # ---- requested metrics ----
        B_corrected = dg$B_corrected,
        max_corr         = as.numeric(max_corr),
        lambda_pop       = dg$lambda_bias,
        Q_art            = dg$Q_art,
        DCS_max          = dg$D_CS_max,
        artifact_flag    = dg$artifact_flag,
        sensitivity_flag = dg$sensitivity_flag,
        reliability_flag = dg$reliability_flag,
         prior_flag = dg$prior_flag,
        max_abs_z = dg$max_abs_z,
        error = NA_character_
      )
      if (n_in  == 0) { rec$replicated_outsample <- 0L; return(rec) }
      if (n_out == 0) { rec$replicated_insample  <- 0L; return(rec) }
        tryCatch({
        cr <- coloc_pair(insample, outsample)
        sm <- cr$summary %>%
          mutate(coloc = PP.H4.abf > pmax(PP.H0.abf, PP.H1.abf, PP.H2.abf, PP.H3.abf))
        s1 <- sm %>% group_by(hit1) %>% summarize(coloc = any(coloc, na.rm = TRUE), .groups = "drop")
        s2 <- sm %>% group_by(hit2) %>% summarize(coloc = any(coloc, na.rm = TRUE), .groups = "drop")
        rec$replicated_insample  <- sum(s1$coloc)
        rec$replicated_outsample <- sum(s2$coloc)
        rec
      }, error = function(e) { rec$error <- conditionMessage(e); rec })
    }

    # ---- run ----
    res <- tryCatch(readRDS(in_file), error = function(e) NULL)
    if (is.null(res) || is.character(res) || !is.list(res) || length(res) == 0) {
      saveRDS(NULL, out_file)
      cat(sprintf("[%s] empty input, wrote NULL\n", gene))
      quit(status = 0)
    }

    recs <- map_dfr(names(res), function(tissue) {
      cat(sprintf("  [%s] tissue: %s\n", gene, tissue))
      flush.console()
      result   <- res[[tissue]]
      insample <- result$insample_res
      if (is.null(insample)) return(NULL)

      mcc <- result$max_cs_corr            # named vector by method; NULL if not stored

    base_recs <- map_dfr(sumstat_methods, function(method) {
            outsample <- result[[method]]
            if (is.null(outsample)) return(NULL)
            one_record(insample, outsample, gene, tissue, method,
                       max_corr = mcc_get(mcc, method))
          })

      # hybrid: SER if R is unreliable, else eb
      eb   <- result$susie_eb_res
      ser  <- result$susie_ser_res
      flag <- eb$R_finite_diagnostics$R_reliability_flag
      hybrid_rec <- NULL
      if (!is.null(eb) && !is.null(ser) && !is.null(flag) && !is.na(flag)) {
        chosen     <- if (isTRUE(flag)) ser else eb
        chosen_nm  <- if (isTRUE(flag)) "susie_ser_res" else "susie_eb_mix_res"
        hybrid_rec <- one_record(insample, chosen, gene, tissue, "eb_or_ser",
                                 max_corr = mcc_get(mcc, chosen_nm))
      }

      bind_rows(base_recs, hybrid_rec)
    })

    saveRDS(recs, out_file)
    cat(sprintf("[%s] saved %d records\n", gene, nrow(recs %||% tibble())))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/"
job_name="coloc"
job_dir="${base_dir}/${job_name}"

# Source: where the per-gene SuSiE result RDS files live
src_dir="${base_dir}/result"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=25000]
#BSUB -W 36:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/18.Fig5B_5D.UKB_Coloc.ipynb" coloc \
    --in-dir SRC_DIR \
    --out-dir WORK_DIR/result/ \
    --mem 40G -J 50
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|SRC_DIR|${src_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"

In [6]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time//code/UKB_benchmark_time.R
# ============================================================
# UKB blood-trait GWAS fine-mapping with KG LD reference
#   - SuSiE-RSS EB-mix (haochen build), two LD inputs:
#       X  = harm$sketch                (sketch factor matrix)
#       LD = get_cormat(harm$sketch)    (explicit correlation matrix)
#   - Profiles ONLY the safe_susie fit step: time / peak / avg memory
#
# Usage: Rscript UKB_fine.R <IDX>
# ============================================================

# ---- batch index from command line ----
args <- commandArgs(trailingOnly = TRUE)
IDX  <- as.integer(args[1])
stopifnot(!is.na(IDX), IDX >= 1)

library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# susieR GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

# ============================================================
# Isolated susieR build (haochen) so its susie_rss + helpers
# stay bound to their own environment.
# ============================================================
source_build <- function(dir) {
  rdir <- if (dir.exists(file.path(dir, "R"))) file.path(dir, "R") else dir
  env  <- new.env(parent = globalenv())
  for (f in list.files(rdir, full.names = TRUE, pattern = "\\.R$"))
    sys.source(f, envir = env)
  env
}

susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

sketch_harmonize <- function(target_data, ref_variants, sketch) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(sketch))
  if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
  sketch2 <- sketch[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
  colnames(sketch2) <- qced$variants_id_original
  list(sumstat = qced, sketch = sketch2,
       flipped_prop = length(flip_index) / ncol(sketch2))
}

# max inter-CS correlation (NA when < 2 CS), against the LD the fit used
cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- resource profiling: wrap ONLY the fit call ----
read_rss_kb <- function(pid = Sys.getpid()) {
  ln <- grep("VmRSS", readLines(sprintf("/proc/%d/status", pid)), value = TRUE)
  as.numeric(sub(".*?([0-9]+).*", "\\1", ln))
}

profile_fit <- function(expr) {                 # expr is lazy -> runs at force()
  pid <- Sys.getpid()
  rss_file <- tempfile(); script <- tempfile(fileext = ".sh")
  writeLines(c("#!/bin/bash",
               sprintf("while true; do awk '/VmRSS/{print $2}' /proc/%d/status >> %s 2>/dev/null; sleep 0.02; done",
                       pid, rss_file)), script)
  base_kb <- read_rss_kb(pid)
  sampler <- as.integer(system(sprintf("bash %s >/dev/null 2>&1 & echo $!", script), intern = TRUE))
  t0  <- Sys.time()
  val <- force(expr)                            # <- the safe_susie(...) step
  t1  <- Sys.time()
  system(sprintf("kill %d", sampler), ignore.stdout = TRUE, ignore.stderr = TRUE)
  Sys.sleep(0.05)
  rss <- suppressWarnings(as.numeric(readLines(rss_file)))
  rss <- c(base_kb, rss[is.finite(rss)])
  list(value         = val,
       time_s        = as.numeric(difftime(t1, t0, units = "secs")),
       peak_mb       = max(rss)  / 1024,
       avg_mb        = mean(rss) / 1024,
       peak_delta_mb = (max(rss) - min(rss)) / 1024,
       n_samples     = length(rss))
}

# ---- config ----
sumstat_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/UKB_blood_real_data/"
snp_info_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/snp_info/"
out_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/result"
ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv"

maf.cut     <- 0.05
missing.cut <- 0.1
B_sketch    <- 10000    # finite-reference size for the KG / sketch panel

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

files <- list.files(sumstat_dir, pattern = ".summary_stats.rds", full.names = TRUE)

idx_start <- 15 * (IDX - 1) + 1
idx_end   <- min(15 * IDX, length(files))

# ============================================================
# Outer loop: regions
# ============================================================
for (i in idx_start:idx_end) {
  tryCatch({
    file      <- files[i]
    base_file <- basename(file)

    # ---- 1. parse region from filename ----
    parts     <- str_split(base_file, "\\.")[[1]]
    blood_chr <- parts[1]
    start     <- parts[2]
    end       <- parts[3]
    chrom     <- str_remove(str_split(blood_chr, "_")[[1]][2], "chr")
    region    <- paste0("chr", chrom, ":", start, "-", end)

    # skip-existing
    res_file <- file.path(out_dir, paste0("res_", region, ".rds"))
    if (file.exists(res_file)) next

    # ---- 2. load sumstats + sufficient stats ----
    suff_stat_file <- str_replace(base_file, "summary", "sufficient")
    sumstat_list   <- readRDS(file)
    suff_stat_list <- readRDS(file.path(sumstat_dir, suff_stat_file))

    if (ncol(sumstat_list$Z) == 0) next

    # dbsnp hg38 lookup (CHR/BP already lifted)
    dpsnp_match <- read_tsv(file.path(snp_info_dir, paste0("chrom_", chrom, "_snp.tsv")),
                            show_col_types = FALSE)

    snp_meta     <- sumstat_list$meta
    target_input <- snp_meta %>%
      left_join(dpsnp_match %>% select(SNP, BP), by = c("ID" = "SNP")) %>%
      select(CHR, BP, ID, REF, ALT, maf) %>%
      mutate(original_ID = paste0(ID, "_", REF)) %>%
      arrange(BP) %>%
      na.omit()

    start  <- min(target_input$BP)
    end    <- max(target_input$BP)
    region <- sprintf("chr%s:%d-%d", chrom, start, end)

    res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
    res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
    sketch_ref           <- res_sketch$ref_panel

    # ---- 3. trait loop ----
    res_susie <- list()

    for (tissue_idx in seq_len(ncol(sumstat_list$Z))) {

      tissue_name <- colnames(sumstat_list$Z)[tissue_idx]
      n           <- suff_stat_list$N
      z           <- sumstat_list$Z[, tissue_idx]

      message(sprintf("[%s]   trait: %s",
                      format(Sys.time(), "%Y-%m-%d %H:%M:%S"), tissue_name))

      # build pos-sorted sumstat with z
      snp_meta <- target_input
      z_df     <- tibble(z = z, original_ID = rownames(sumstat_list$Z))
      snp_meta <- left_join(snp_meta, z_df, by = "original_ID")
      colnames(snp_meta) <- c("chrom", "pos", "snp", "A2", "A1", "maf", "original_ID", "z")

      # ---- allele QC against KG reference ----
      harm <- sketch_harmonize(target_data  = snp_meta,
                               ref_variants = sketch_ref %>% select(-variant_id),
                               sketch       = res_sketch$LD_matrix)

      sketch_LD      <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)
      krig           <- kriging_fix(z = harm$sumstat$z, LD = sketch_LD, n = n)
      harm$sumstat$z <- krig$z

      if (ncol(harm$sketch) == 0) {
        message(sprintf("  No variants survive harmonization for trait %s - skipping", tissue_name))
        next
      }

      # ==========================================================
      # susie_eb_mix_res, two LD inputs. Profile ONLY the safe_susie step.
      #   (X)  X = harm$sketch
      #   (LD) R = get_cormat(harm$sketch)   (built OUTSIDE the profiled region)
      # ==========================================================
        LD_sketch <- get_cormat(harm$sketch)       # built OUTSIDE the profiled region

      prof_rss_X  <- profile_fit(
        safe_susie(susie_hc, "susie_rss_res_X",
                   z = harm$sumstat$z, X = harm$sketch, n = n))

      prof_rss_LD <- profile_fit(
        safe_susie(susie_hc, "susie_rss_res_LD",
                   z = harm$sumstat$z, R = LD_sketch, n = n))

      prof_X      <- profile_fit(
        safe_susie(susie_hc, "susie_eb_mix_res_X",
                   z = harm$sumstat$z, X = harm$sketch, n = n,
                   R_finite = B_sketch, R_mismatch = "eb_mix"))

      prof_LD     <- profile_fit(
        safe_susie(susie_hc, "susie_eb_mix_res_LD",
                   z = harm$sumstat$z, R = LD_sketch, n = n,
                   R_finite = B_sketch, R_mismatch = "eb_mix"))

      # helper: number of credible sets (NA if fit failed)
      ncs <- function(fit) if (is.null(fit)) NA_integer_ else length(fit$sets$cs)

      max_abs_z <- max(abs(harm$sumstat$z), na.rm = TRUE)   # per-trait, on the kriging-fixed z

      bench <- tibble(
        tissue        = tissue_name,
        method        = c("rss_X", "rss_LD", "eb_mix_X", "eb_mix_LD"),
        n_cs          = c(ncs(prof_rss_X$value),      ncs(prof_rss_LD$value),      ncs(prof_X$value),          ncs(prof_LD$value)),
        max_abs_z     = max_abs_z,
        time_s        = c(prof_rss_X$time_s,          prof_rss_LD$time_s,          prof_X$time_s,              prof_LD$time_s),
        peak_mb       = c(prof_rss_X$peak_mb,         prof_rss_LD$peak_mb,         prof_X$peak_mb,             prof_LD$peak_mb),
        avg_mb        = c(prof_rss_X$avg_mb,          prof_rss_LD$avg_mb,          prof_X$avg_mb,              prof_LD$avg_mb),
        peak_delta_mb = c(prof_rss_X$peak_delta_mb,   prof_rss_LD$peak_delta_mb,   prof_X$peak_delta_mb,       prof_LD$peak_delta_mb),
        n_samples     = c(prof_rss_X$n_samples,       prof_rss_LD$n_samples,       prof_X$n_samples,           prof_LD$n_samples)
      )

      max_cs_corr <- c(
        rss_X     = cs_maxcor(prof_rss_X$value,  LD_sketch),
        rss_LD    = cs_maxcor(prof_rss_LD$value, LD_sketch),
        eb_mix_X  = cs_maxcor(prof_X$value,      LD_sketch),
        eb_mix_LD = cs_maxcor(prof_LD$value,     LD_sketch)
      )

      res_susie[[tissue_name]] <- list(
        variant_ids            = colnames(harm$sketch),
        susie_rss_res_X        = prof_rss_X$value,
        susie_rss_res_LD       = prof_rss_LD$value,
        susie_eb_mix_res_X     = prof_X$value,
        susie_eb_mix_res_LD    = prof_LD$value,
        bench                  = bench,           # time / peak / avg per method
        max_cs_corr            = max_cs_corr,
        sample_size            = n,
        flipped_prop           = harm$flipped_prop,
        kriging_outlier_number = krig$outlier_number,
        original_file_path     = file,
        region                 = region
      )
    }

    # ---- save per-region ----
    if (length(res_susie) == 0) {
      saveRDS("no traits processed",
              file.path(out_dir, paste0("nores_", region, ".rds")))
    } else {
      saveRDS(res_susie, res_file)
    }
  }, error = function(e) {
    cat(sprintf("[FAIL] region i=%d (%s): %s\n",
                i, basename(files[i]), conditionMessage(e)))
  })
}

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time//code/UKB_benchmark_time.R


In [5]:
%%writefile  /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/code/submit_UKB_benchmark.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J "UKB_bench[1-65]"
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/log/UKB_bench.%J.%I.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/log/UKB_bench.%J.%I.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/
mkdir -p ${workdir}/log
code_dir="$workdir/code"
tmp_root="$workdir/tmp"

tmp_dir="$tmp_root/${LSB_JOBID}_${LSB_JOBINDEX}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

Rscript "$code_dir/UKB_benchmark_time.R" "$LSB_JOBINDEX"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_benchmark_time/code/submit_UKB_benchmark.sh
